# Regressão com RNA - Wine Quality Dataset

Implementação de **regressão usando Rede Neural Artificial (RNA)** com o dataset **Wine Quality**, seguindo a arquitetura vista na aula **FRA - Aula 12 - 2.5 Resolução de exercício de RNA - Regressão**.

**Base de dados:** Wine Quality Dataset (Red Wine)  
**Descrição:** O objetivo deste dataset é prever a qualidade dos vinhos com base em suas características químicas. A variável target (y) é o score de qualidade do vinho, que varia de 0 (pior qualidade) a 10 (melhor qualidade).  
**Tamanho:** 1599 amostras, 12 features (11 características químicas + 1 variável alvo)

## 1. Importação das bibliotecas

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

print(f"TensorFlow version: {tf.__version__}")

## 2. Carregar a base de dados Wine Quality

In [ ]:
# Carregar o dataset de vinhos tintos
url = "https://archive.ics.uci.edu/ml/machine-learning-databases/wine-quality/winequality-red.csv"
data = pd.read_csv(url, delimiter=';')

print(f"Dimensões do dataset: {data.shape}")
print(f"\nPrimeiras linhas do dataset:")
data.head()

In [ ]:
# Renomear colunas para português
data.columns = [
    'acidez_fixa',                  # fixed acidity
    'acidez_volatil',               # volatile acidity
    'acido_citrico',                # citric acid
    'acucar_residual',              # residual sugar
    'cloretos',                     # chlorides
    'dioxido_de_enxofre_livre',     # free sulfur dioxide
    'dioxido_de_enxofre_total',     # total sulfur dioxide
    'densidade',                    # density
    'pH',                           # pH
    'sulfatos',                     # sulphates
    'alcool',                       # alcohol
    'score_qualidade_vinho'         # quality
]

print("Colunas renomeadas com sucesso!")
print(f"\nNomes das colunas: {list(data.columns)}")
data.head()

## 3. Análise Exploratória dos Dados

In [ ]:
# Estatísticas descritivas
print("=== Estatísticas Descritivas ===")
data.describe()

In [ ]:
# Verificar valores nulos
print("=== Valores Nulos por Coluna ===")
print(data.isnull().sum())
print(f"\nTotal de valores nulos: {data.isnull().sum().sum()}")

In [ ]:
# Distribuição da variável target (score de qualidade)
plt.figure(figsize=(10, 5))

plt.subplot(1, 2, 1)
data['score_qualidade_vinho'].hist(bins=10, edgecolor='black', alpha=0.7)
plt.title('Distribuição do Score de Qualidade', fontsize=12, fontweight='bold')
plt.xlabel('Score de Qualidade')
plt.ylabel('Frequência')

plt.subplot(1, 2, 2)
data['score_qualidade_vinho'].value_counts().sort_index().plot(kind='bar', edgecolor='black', alpha=0.7)
plt.title('Contagem por Score de Qualidade', fontsize=12, fontweight='bold')
plt.xlabel('Score de Qualidade')
plt.ylabel('Contagem')

plt.tight_layout()
plt.show()

print(f"\nDistribuição dos scores:")
print(data['score_qualidade_vinho'].value_counts().sort_index())

## 4. Separação das Features (X) e Target (y)

In [ ]:
# Separar features (X) e target (y)
# A última coluna (índice -1) é a variável target (score_qualidade_vinho)
X = data.iloc[:, :-1].values  # Todas as colunas exceto a última
y = data.iloc[:, -1].values   # Última coluna (score de qualidade)

print(f"Formato de X (features): {X.shape}")
print(f"Formato de y (target): {y.shape}")
print(f"\nFeatures utilizadas: {list(data.columns[:-1])}")
print(f"\nExemplo de X[0]: {X[0]}")
print(f"Exemplo de y[0]: {y[0]}")

## 5. Divisão em Treino e Teste

In [ ]:
# Dividir em conjunto de treino (80%) e teste (20%)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print(f"Dados de treino: X={X_train.shape}, y={y_train.shape}")
print(f"Dados de teste: X={X_test.shape}, y={y_test.shape}")

## 6. Pré-processamento: Normalização dos Dados

A normalização é essencial em redes neurais para que todas as features tenham a mesma escala. Utilizamos o `StandardScaler` para padronizar os dados (média = 0, desvio padrão = 1).

In [ ]:
# Normalizar as features usando StandardScaler
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

print("Dados normalizados com sucesso!")
print(f"\nEstatísticas após normalização (treino):")
print(f"  Média por feature: {X_train.mean(axis=0).round(4)}")
print(f"  Desvio padrão por feature: {X_train.std(axis=0).round(4)}")

## 7. Construção da Arquitetura RNA para Regressão

A arquitetura da rede neural utilizada para regressão:

1. **Dense (64 neurônios)**: Primeira camada oculta com ativação ReLU
2. **Dense (32 neurônios)**: Segunda camada oculta com ativação ReLU
3. **Dense (1 neurônio)**: Camada de saída **sem ativação** (linear) para prever o valor contínuo do score

**Diferenças em relação à classificação:**
- A camada de saída tem apenas 1 neurônio (para prever um valor único)
- Não há função de ativação na saída (ativação linear para valores contínuos)
- A função de perda é **MSE (Mean Squared Error)** em vez de crossentropy

In [ ]:
# Construir o modelo de regressão
model = tf.keras.Sequential([
    tf.keras.layers.Dense(64, activation='relu', input_shape=(X_train.shape[1],)),
    tf.keras.layers.Dense(32, activation='relu'),
    tf.keras.layers.Dense(1)  # Saída linear para regressão
])

# Compilar o modelo com otimizador Adam e função de perda MSE
model.compile(
    optimizer='adam',
    loss='mse',  # Mean Squared Error - padrão para regressão
    metrics=['mae']  # Mean Absolute Error como métrica adicional
)

model.summary()

## 8. Treinamento do Modelo

In [ ]:
# Treinar o modelo
history = model.fit(
    X_train,
    y_train,
    epochs=100,
    validation_split=0.2,
    batch_size=32,
    verbose=1
)

## 9. Gráficos de Avaliação do Modelo (Loss)

In [ ]:
history_dict = history.history

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Gráfico de Loss (MSE)
axes[0].plot(history_dict['loss'], 'b-', label='Treino', linewidth=2)
axes[0].plot(history_dict['val_loss'], 'r-', label='Validação', linewidth=2)
axes[0].set_title('Curva de Perda (Loss - MSE)', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Época', fontsize=12)
axes[0].set_ylabel('MSE (Mean Squared Error)', fontsize=12)
axes[0].legend(fontsize=11)
axes[0].grid(True, alpha=0.3)

# Gráfico de MAE
axes[1].plot(history_dict['mae'], 'b-', label='Treino', linewidth=2)
axes[1].plot(history_dict['val_mae'], 'r-', label='Validação', linewidth=2)
axes[1].set_title('Curva de MAE (Mean Absolute Error)', fontsize=14, fontweight='bold')
axes[1].set_xlabel('Época', fontsize=12)
axes[1].set_ylabel('MAE', fontsize=12)
axes[1].legend(fontsize=11)
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

---

## Explicação dos Gráficos de Perda (Loss)

### Gráfico de MSE (Mean Squared Error)

O gráfico de **MSE (Erro Quadrático Médio)** mostra como o erro do modelo evolui ao longo das épocas de treinamento. O MSE é a função de perda padrão para problemas de regressão.

**O que é MSE?**
- O MSE calcula a média dos quadrados das diferenças entre os valores previstos e os valores reais
- Fórmula: $MSE = \frac{1}{n} \sum_{i=1}^{n} (y_i - \hat{y}_i)^2$
- Valores menores indicam melhor performance
- Por elevar ao quadrado, penaliza mais fortemente os erros grandes

**Análise do comportamento:**
- **Curva de Treino (azul):** Decresce rapidamente nas primeiras épocas e depois estabiliza. Isso indica que o modelo está aprendendo a ajustar seus pesos para minimizar o erro.
- **Curva de Validação (vermelha):** Também apresenta tendência de queda, acompanhando a curva de treino. Quando ambas as curvas diminuem juntas, é um bom sinal de que o modelo está generalizando bem.

**Interpretação:**
- Se a perda de validação começasse a subir enquanto a de treino continuasse caindo, indicaria **overfitting**.
- A convergência e proximidade das curvas demonstra que o modelo está aprendendo padrões generalizáveis.

---

### Gráfico de MAE (Mean Absolute Error)

O gráfico de **MAE (Erro Absoluto Médio)** é uma métrica complementar mais interpretável.

**O que é MAE?**
- O MAE calcula a média das diferenças absolutas entre os valores previstos e os valores reais
- Fórmula: $MAE = \frac{1}{n} \sum_{i=1}^{n} |y_i - \hat{y}_i|$
- É mais fácil de interpretar: representa o erro médio na mesma escala da variável target
- Por exemplo, um MAE de 0.5 significa que, em média, o modelo erra 0.5 pontos no score de qualidade

**Análise do comportamento:**
- Segue um padrão similar ao MSE, com queda inicial rápida e estabilização.
- A proximidade entre treino e validação confirma boa generalização.

---

## 10. Avaliação no Conjunto de Teste

In [ ]:
# Fazer previsões no conjunto de teste
y_pred = model.predict(X_test).flatten()

# Calcular métricas de avaliação
mae = mean_absolute_error(y_test, y_pred)
mse = mean_squared_error(y_test, y_pred)
rmse = np.sqrt(mse)
r2 = r2_score(y_test, y_pred)

print("="*50)
print("    MÉTRICAS DE AVALIAÇÃO DO MODELO")
print("="*50)
print(f"\n  MAE  (Mean Absolute Error):    {mae:.4f}")
print(f"  MSE  (Mean Squared Error):     {mse:.4f}")
print(f"  RMSE (Root Mean Squared Error): {rmse:.4f}")
print(f"  R²   (Coeficiente de Determinação): {r2:.4f}")
print("\n" + "="*50)

---

## Explicação das Métricas de Avaliação

### MAE (Mean Absolute Error - Erro Absoluto Médio)

**O que é:** Média das diferenças absolutas entre os valores previstos e reais.

**Fórmula:** $MAE = \frac{1}{n} \sum_{i=1}^{n} |y_i - \hat{y}_i|$

**Interpretação:**
- Mede o erro médio em unidades da variável target (score de qualidade 0-10)
- Um MAE de 0.5 significa que, em média, as previsões diferem 0.5 pontos do valor real
- Mais robusto a outliers que o MSE
- **Quanto menor, melhor!**

---

### MSE (Mean Squared Error - Erro Quadrático Médio)

**O que é:** Média dos quadrados das diferenças entre valores previstos e reais.

**Fórmula:** $MSE = \frac{1}{n} \sum_{i=1}^{n} (y_i - \hat{y}_i)^2$

**Interpretação:**
- Penaliza mais fortemente os erros grandes (por elevar ao quadrado)
- A unidade é o quadrado da variável target (difícil de interpretar diretamente)
- Muito utilizado como função de perda no treinamento
- **Quanto menor, melhor!**

---

### RMSE (Root Mean Squared Error - Raiz do Erro Quadrático Médio)

**O que é:** Raiz quadrada do MSE.

**Fórmula:** $RMSE = \sqrt{MSE}$

**Interpretação:**
- Retorna o erro para a mesma escala da variável target
- Mantém a propriedade de penalizar erros grandes
- **Quanto menor, melhor!**

---

### R² (Coeficiente de Determinação)

**O que é:** Proporção da variância dos dados que é explicada pelo modelo.

**Fórmula:** $R^2 = 1 - \frac{\sum_{i=1}^{n}(y_i - \hat{y}_i)^2}{\sum_{i=1}^{n}(y_i - \bar{y})^2}$

**Interpretação:**
- Varia de 0 a 1 (pode ser negativo em casos ruins)
- **R² = 1:** Modelo perfeito (explica 100% da variância)
- **R² = 0:** Modelo igual à média (não aprendeu nada)
- **R² < 0:** Modelo pior que simplesmente prever a média
- Um R² de 0.4, por exemplo, indica que o modelo explica 40% da variância dos dados
- **Quanto maior (mais próximo de 1), melhor!**

---

## 11. Visualização: Valores Reais vs Previstos

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Gráfico de dispersão: Real vs Previsto
axes[0].scatter(y_test, y_pred, alpha=0.5, edgecolors='black', linewidth=0.5)
axes[0].plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--', linewidth=2, label='Linha Ideal')
axes[0].set_xlabel('Valor Real', fontsize=12)
axes[0].set_ylabel('Valor Previsto', fontsize=12)
axes[0].set_title('Valores Reais vs Previstos', fontsize=14, fontweight='bold')
axes[0].legend(fontsize=11)
axes[0].grid(True, alpha=0.3)

# Histograma dos resíduos (erros)
residuos = y_test - y_pred
axes[1].hist(residuos, bins=30, edgecolor='black', alpha=0.7)
axes[1].axvline(x=0, color='r', linestyle='--', linewidth=2, label='Erro Zero')
axes[1].set_xlabel('Resíduo (Real - Previsto)', fontsize=12)
axes[1].set_ylabel('Frequência', fontsize=12)
axes[1].set_title('Distribuição dos Resíduos', fontsize=14, fontweight='bold')
axes[1].legend(fontsize=11)
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f"Estatísticas dos resíduos:")
print(f"  Média: {residuos.mean():.4f} (ideal: próximo de 0)")
print(f"  Desvio padrão: {residuos.std():.4f}")

### Interpretação dos gráficos:

**Gráfico de Dispersão (Real vs Previsto):**
- Os pontos idealmente deveriam estar sobre a linha vermelha tracejada
- Quanto mais próximos da linha, melhores são as previsões
- Dispersão vertical indica variabilidade nas previsões

**Histograma dos Resíduos:**
- Idealmente, os resíduos devem ter distribuição normal centrada em zero
- Uma distribuição simétrica em torno de zero indica que o modelo não tem viés sistemático

## 12. Exemplos de Previsões

In [ ]:
# Mostrar algumas previsões com os valores reais
print("Exemplos de previsões do modelo:")
print("-" * 50)
print(f"{'Índice':^8} | {'Real':^10} | {'Previsto':^10} | {'Erro':^10}")
print("-" * 50)

# Selecionar 15 amostras aleatórias
np.random.seed(42)
indices = np.random.choice(len(y_test), 15, replace=False)

for idx in indices:
    real = y_test[idx]
    pred = y_pred[idx]
    erro = real - pred
    print(f"{idx:^8} | {real:^10.1f} | {pred:^10.2f} | {erro:^+10.2f}")

print("-" * 50)

## 13. Conclusão

Neste notebook, implementamos uma **Rede Neural Artificial (RNA) para Regressão** usando o dataset Wine Quality. Os principais pontos abordados foram:

### Resumo da Implementação:

1. **Dataset:** Wine Quality com 1599 amostras e 11 características químicas para prever o score de qualidade (0-10)

2. **Arquitetura da RNA:**
   - Camada de entrada com 11 features
   - Camada oculta com 64 neurônios (ReLU)
   - Camada oculta com 32 neurônios (ReLU)
   - Camada de saída com 1 neurônio (linear)

3. **Pré-processamento:**
   - Normalização com StandardScaler para uniformizar as escalas
   - Divisão 80% treino / 20% teste

4. **Treinamento:**
   - Otimizador: Adam
   - Função de perda: MSE (Mean Squared Error)
   - 100 épocas com batch size de 32

### Diferenças entre Regressão e Classificação:

| Aspecto | Regressão | Classificação |
|---------|-----------|---------------|
| Saída | Valor contínuo | Classe/categoria |
| Neurônio de saída | 1 (linear) | N classes (softmax) |
| Função de perda | MSE, MAE | CrossEntropy |
| Métricas | MAE, MSE, R² | Acurácia, F1-Score |